In [ ]:
import os, json, time, random, hashlib, platform, warnings
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, torch, sdv, optuna
from scipy.stats import ks_2samp
from sdv.metadata import Metadata
from sdv.single_table import TVAESynthesizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, matthews_corrcoef
from sklearn.model_selection import train_test_split

CFG = dict(
    run_name="tvae_perclass_final_v2",
    data_dir="../datasets/final_datasets_v2/final datasets",
    out_dir="../outputs/tvae_final_v2",
    fixed_params=dict(epochs=300, batch_size=1000, embedding_dim=64, dims="128x128",
                      l2scale=0.0007235103476873236, loss_factor=1),
    seeds=dict(split=11, generator=22, judge=33),
    val_fraction=0.2,
    min_code_count=3,
    log_threshold=1000,
    n_trials=1, trial_rows_per_class=1500, trial_timeout_min=150,
)
TRAIN_FILE = f"{CFG['data_dir']}/generators/TVAE/unsw_nb15_attacks_smote_10k.csv"
OUT = CFG["out_dir"]; os.makedirs(f"{OUT}/models", exist_ok=True)
TARGET, CONST = "Attack", "Label"
CODE_COLS = ["PROTOCOL", "L7_PROTO", "TCP_FLAGS", "CLIENT_TCP_FLAGS", "SERVER_TCP_FLAGS",
             "DNS_QUERY_TYPE", "FTP_COMMAND_RET_CODE"]
GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "none", "| sdv", sdv.__version__, "| optuna", optuna.__version__)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if GPU: torch.cuda.manual_seed_all(s)

In [ ]:
raw = pd.read_csv(TRAIN_FILE)
FEATS = [c for c in raw.columns if c not in (TARGET, CONST)]

def clean(df, med=None):
    X = df[FEATS].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    med = X.median() if med is None else med
    return X.fillna(med).fillna(0), med

X_all, MED = clean(raw)
y_all = raw[TARGET].astype(str)
CLASSES = sorted(y_all.unique())

VALID = {}
for c in CODE_COLS:
    vc = X_all[c].value_counts()
    VALID[c] = np.sort(vc[vc >= CFG["min_code_count"]].index.values.astype(float))

def snap(X):
    X = X.copy()
    for c in CODE_COLS:
        v, x = VALID[c], X[c].values.astype(float)
        if len(v) == 1: X[c] = v[0]; continue
        i = np.clip(np.searchsorted(v, x), 1, len(v) - 1)
        X[c] = np.where(np.abs(x - v[i - 1]) <= np.abs(v[i] - x), v[i - 1], v[i])
    return X

def invented_rows(X):
    return int(np.any([~np.isin(X[c].values.astype(float), VALID[c]) for c in CODE_COLS], axis=0).sum())

print(f"rows {len(raw):,} | rows with an invented code value before snapping: {invented_rows(X_all):,}")
X_all = snap(X_all)
print("after snapping:", invented_rows(X_all))
print(y_all.value_counts().to_string())

In [ ]:
dfa = pd.concat([X_all, y_all.rename(TARGET)], axis=1).drop_duplicates().reset_index(drop=True)
tr, va = train_test_split(dfa, test_size=CFG["val_fraction"], stratify=dfa[TARGET], random_state=CFG["seeds"]["split"])
X_tr, y_tr = tr[FEATS].reset_index(drop=True), tr[TARGET].reset_index(drop=True)
X_va, y_va = va[FEATS].reset_index(drop=True), va[TARGET].reset_index(drop=True)
print(f"unique rows {len(dfa):,} | train {len(X_tr):,} | validation {len(X_va):,}")

INT_COLS = [c for c in FEATS if np.all(np.mod(X_tr[c], 1) == 0)]
LOG_COLS = [c for c in FEATS if c not in CODE_COLS and X_tr[c].min() >= 0 and X_tr[c].max() > CFG["log_threshold"]]
COL_MIN, COL_MAX = X_tr.min(), X_tr.max()
print(f"{len(INT_COLS)} integer columns, {len(LOG_COLS)} log1p columns")

In [ ]:
def to_model(X):
    Z = X.copy()
    for c in LOG_COLS: Z[c] = np.log1p(Z[c])
    for c in CODE_COLS: Z[c] = Z[c].map(lambda v: f"{v:g}")
    return Z

def from_model(Z):
    X = Z.copy()
    for c in CODE_COLS: X[c] = pd.to_numeric(X[c])
    for c in LOG_COLS: X[c] = np.expm1(X[c])
    X = X[FEATS].clip(lower=COL_MIN, upper=COL_MAX, axis=1)
    X[INT_COLS] = X[INT_COLS].round()
    return X

def make_tvae(Z, p):
    md_ = Metadata.detect_from_dataframes({"t": Z}, infer_keys=None)
    for c in Z.columns:
        md_.update_column(c, table_name="t", sdtype="categorical" if c in CODE_COLS else "numerical")
    dims = tuple(int(d) for d in p["dims"].split("x"))
    return TVAESynthesizer(md_, epochs=p["epochs"], batch_size=p["batch_size"], embedding_dim=p["embedding_dim"],
                           compress_dims=dims, decompress_dims=dims, l2scale=p["l2scale"], loss_factor=p["loss_factor"],
                           enforce_min_max_values=True, enforce_rounding=False, enable_gpu=GPU, verbose=False)

def generate(X, y, p, n_per_class, seed, keep_models=False):
    parts, models, losses = [], {}, {}
    for c in CLASSES:
        Z = to_model(X[y == c].reset_index(drop=True))
        set_seed(seed); m = make_tvae(Z, p); m.fit({"t": Z})
        set_seed(seed); s = m.sample(num_rows=int(n_per_class[c])); s = s["t"] if isinstance(s, dict) else s
        s = from_model(s); s[TARGET] = c; parts.append(s)
        try: losses[c] = m.get_loss_values()
        except Exception: pass
        if keep_models: models[c] = m
    return pd.concat(parts, ignore_index=True), models, losses

In [ ]:
def rf(seed): return RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=seed, class_weight="balanced_subsample")

def ks_mean(S, X_ref, y_ref, cap=2000):
    vals = []
    for c in CLASSES:
        a = X_ref[y_ref == c]; b = S[S[TARGET] == c][FEATS]
        if len(b) == 0: continue
        a = a.sample(min(cap, len(a)), random_state=0); b = b.sample(min(cap, len(b)), random_state=0)
        vals += [ks_2samp(a[f], b[f]).statistic for f in FEATS]
    return float(np.mean(vals))

def evaluate(S, judge, X_ref, y_ref, X_test, y_test, trtr_f1):
    Xs, ys = S[FEATS], S[TARGET].values
    pred = judge.predict(Xs)
    per_class = {c: float((pred[ys == c] == c).mean()) for c in CLASSES}
    t = rf(CFG["seeds"]["judge"]).fit(Xs, ys); p = t.predict(X_test)
    tstr = f1_score(y_test, p, average="macro")
    return dict(label_match_macro=float(np.mean(list(per_class.values()))), label_match_per_class=per_class,
                tstr_macro_f1=float(tstr), tstr_mcc=float(matthews_corrcoef(y_test, p)),
                ml_efficacy=float(tstr / trtr_f1),
                tstr_f1_per_class=dict(zip(CLASSES, map(float, f1_score(y_test, p, average=None, labels=CLASSES)))),
                negative_cells=int((Xs < 0).sum().sum()), invented_code_rows=invented_rows(Xs),
                missing_cells=int(Xs.isna().sum().sum()), ks_mean=ks_mean(S, X_ref, y_ref))

judge = rf(CFG["seeds"]["judge"]).fit(X_tr, y_tr)
TRTR_VAL = f1_score(y_va, judge.predict(X_va), average="macro")
print(f"TRTR macro-F1 on internal validation: {TRTR_VAL:.4f}")

In [ ]:
def sub(X, y, k, seed):
    idx = np.concatenate([y.index[y == c].to_series().sample(min(k, int((y == c).sum())), random_state=seed).values
                          for c in CLASSES])
    return X.loc[idx].reset_index(drop=True), y.loc[idx].reset_index(drop=True)

Xs_tr, ys_tr = sub(X_tr, y_tr, CFG["trial_rows_per_class"], CFG["seeds"]["split"])
Xs_va, ys_va = sub(X_va, y_va, CFG["trial_rows_per_class"] // 2, CFG["seeds"]["split"])
judge_sub = rf(CFG["seeds"]["judge"]).fit(Xs_tr, ys_tr)
TRTR_SUB = f1_score(ys_va, judge_sub.predict(Xs_va), average="macro")
n_sub = ys_tr.value_counts()

def objective(trial):
    p = dict(epochs=trial.suggest_categorical("epochs", [50, 100, 200, 300]),
             batch_size=trial.suggest_categorical("batch_size", [100, 250, 500, 1000]),
             embedding_dim=trial.suggest_categorical("embedding_dim", [64, 128, 256]),
             dims=trial.suggest_categorical("dims", ["128x128", "256x256", "256x256x256"]),
             l2scale=trial.suggest_float("l2scale", 1e-6, 1e-3, log=True),
             loss_factor=trial.suggest_categorical("loss_factor", [1, 2, 3, 5]))
    t0 = time.time()
    S, _, _ = generate(Xs_tr, ys_tr, p, n_sub, CFG["seeds"]["generator"])
    r = evaluate(S, judge_sub, Xs_tr, ys_tr, Xs_va, ys_va, TRTR_SUB)
    for k in ("label_match_macro", "tstr_macro_f1", "ks_mean", "negative_cells", "invented_code_rows"):
        trial.set_user_attr(k, r[k])
    trial.set_user_attr("minutes", round((time.time() - t0) / 60, 1))
    return (r["label_match_macro"] + r["tstr_macro_f1"]) / 2

study = optuna.create_study(study_name=CFG["run_name"], direction="maximize", load_if_exists=True,
                            storage=f"sqlite:///{OUT}/optuna_tvae.db",
                            sampler=optuna.samplers.TPESampler(seed=CFG["seeds"]["generator"]))
if len(study.trials) == 0:
    study.enqueue_trial(CFG.get("fixed_params") or
                        dict(epochs=300, batch_size=500, embedding_dim=128, dims="128x128", l2scale=1e-5, loss_factor=2))
done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
save_trials = lambda st, tr: st.trials_dataframe().to_csv(f"{OUT}/optuna_trials.csv", index=False)
study.optimize(objective, n_trials=max(0, CFG["n_trials"] - done), timeout=CFG["trial_timeout_min"] * 60,
               callbacks=[save_trials], catch=(Exception,))
save_trials(study, None)
cols = ["number", "value", "user_attrs_label_match_macro", "user_attrs_tstr_macro_f1", "user_attrs_ks_mean",
        "user_attrs_minutes"] + [c for c in study.trials_dataframe().columns if c.startswith("params_")]
display(study.trials_dataframe()[cols].sort_values("value", ascending=False))
BEST = study.best_trial.params
print("best:", BEST, "| objective", round(study.best_value, 4), "| trial 0:", round(study.trials[0].value or float('nan'), 4))

In [ ]:
t0 = time.time()
SYN, MODELS, LOSSES = generate(X_tr, y_tr, BEST, y_all.value_counts(), CFG["seeds"]["generator"], keep_models=True)
fit_min = round((time.time() - t0) / 60, 1)
SYN[CONST] = 1
SYN = SYN[list(raw.columns)]
syn_path = f"{OUT}/synthetic_tvae_perclass_smote_10k.csv"
SYN.to_csv(syn_path, index=False)
for c, m in MODELS.items(): m.save(f"{OUT}/models/tvae_{c}.pkl")
pd.concat({c: l for c, l in LOSSES.items()}, names=["class"]).to_csv(f"{OUT}/loss_per_class.csv")
print(f"{len(SYN):,} rows in {fit_min} min ->", syn_path)
print(SYN[TARGET].value_counts().to_string())

In [ ]:
Xs, ys = SYN[FEATS], SYN[TARGET].values
pred = judge.predict(Xs)
label_match = {c: float((pred[ys == c] == c).mean()) for c in CLASSES}
QUALITY = dict(
    rows_requested=y_all.value_counts().to_dict(), rows_generated=SYN[TARGET].value_counts().to_dict(),
    label_match_macro=float(np.mean(list(label_match.values()))), label_match_per_class=label_match,
    ks_mean=ks_mean(SYN, X_tr, y_tr),
    negative_cells=int((Xs < 0).sum().sum()), invented_code_rows=invented_rows(Xs), missing_cells=int(Xs.isna().sum().sum()),
)
print(pd.Series({k: v for k, v in QUALITY.items() if not isinstance(v, dict)}).round(4).to_string())
print(pd.DataFrame({"requested": QUALITY["rows_requested"], "generated": QUALITY["rows_generated"],
                    "label_match": label_match}).round(3).to_string())

In [ ]:
manifest = dict(
    run_name=CFG["run_name"], created_utc=pd.Timestamp.utcnow().isoformat(timespec="seconds"), config=CFG,
    config_sha256=hashlib.sha256(json.dumps(CFG, sort_keys=True).encode()).hexdigest(),
    inputs={"train_file": os.path.basename(TRAIN_FILE), "train_sha256": sha256(TRAIN_FILE)},
    best_params=BEST, optuna={"trials_complete": len([t for t in study.trials if t.value is not None]),
                              "best_value": study.best_value, "default_trial_value": study.trials[0].value},
    output={"file": os.path.basename(syn_path), "sha256": sha256(syn_path), "rows": len(SYN),
            "class_counts": SYN[TARGET].value_counts().to_dict(), "fit_minutes": fit_min},
    quality=QUALITY,
    environment={"python": platform.python_version(), "sdv": sdv.__version__, "torch": torch.__version__,
                 "optuna": optuna.__version__, "gpu": torch.cuda.get_device_name(0) if GPU else None},
)
with open(f"{OUT}/manifest_{CFG['run_name']}.json", "w", encoding="utf8") as f: json.dump(manifest, f, indent=2)
print("saved", f"{OUT}/manifest_{CFG['run_name']}.json")